# The virtual quantum bench — tuning loop (M2)

This notebook is the *interactive* face of the platform: build a device, run it, **vary a knob, watch the metric respond**, and let the optimizer find the operating point — exactly the skill a researcher practices on a real bench, but virtual and free.

It ties together everything:
- **M0** multi-rate engine (faint pulses + realistic impairments, validated vs a brute-force reference),
- **M1** decoy-BB84 + the Rusca-2018 **finite-key** secret-key rate (validated vs the published figure),
- **M2** the **sweep/optimize** harness, declarative **scenarios**, and the **QRNG** plugin (same kernel, different domain).

Run top-to-bottom (`pip install -e .` first).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qsim.qkd.reference import build_decoy_bb84, load_default_profile
from qsim.qkd.keyrate import skr_from_rates, skr_for_params
from qsim.qkd.channel import ChannelParams
from qsim.core.sweep import sweep, optimize

prof = load_default_profile()
MU1, MU2, P1, PZ, REP = (prof.value('mu1'), prof.value('mu2'),
                         prof.value('p_mu1'), prof.value('p_Z'), prof.value('rep_rate'))
print('profile:', prof.name, '| mu1', MU1, 'mu2', MU2, '| rep_rate', REP, 'Hz')

## 1. Build & run the device, read its proven key rate

The multi-rate engine runs the full impairment set (afterpulsing, slow AMZI phase drift, dead time). The detector accumulates the per-(basis, intensity) sifted counts; the Rusca finite-key bound turns them into a *proven* secret-key rate at a chosen privacy-amplification block size.

In [ ]:
_g, sched, det, amzi = build_decoy_bb84(prof, length_km=25.0, locked=True)
sched.run(n_ticks=300, dt_slow=1e-3, pulses_per_tick=200_000, rng=np.random.default_rng(1))
rates = det.rates()
skr, kr = skr_from_rates(rates, mu1=MU1, mu2=MU2, p_mu1=P1, p_Z=PZ, n_Z_block=1e9, rep_rate=REP)
print(f"QBER_Z={rates['E_Z1']*100:.2f}%  QBER_X={rates['E_X1']*100:.2f}%  phi_Z={kr.phiZ_u*100:.2f}%")
print(f"proven SKR = {skr:.3e} Hz   (l/n_Z = {kr.length/kr.nZ:.3f},  feasible={kr.feasible})")

## 2. The tuning loop — sweep the source intensity μ₁

The classic decoy-state tuning task: there is an *optimal* mean photon number. Too low and the key rate starves; too high and multi-photon pulses leak information (the finite-key bound punishes it). We sweep the analytic finite-key model (fast) at a fixed loss.

In [ ]:
PARAMS = ChannelParams(eta_det=0.5, p_dc=5e-9, e_d=0.01, tau_dead=100e-9)
LOSS_DB, N_Z = 30.0, 1e9

def run(mu1, p_Z, mu2=0.1, p_mu1=0.7):
    s, _ = skr_for_params(LOSS_DB, N_Z, mu1, mu2, p_mu1, p_Z, rep_rate=1e9,
                          params=PARAMS, eps_sec=1e-9, eps_cor=1e-15, f_ec=1.16)
    return {'skr': s}

s1 = sweep(run, {'mu1': list(np.round(np.linspace(0.15, 0.9, 40), 4))}, fixed={'p_Z': 0.9})
best, best_skr = s1.best('skr')
print(f"optimal mu1 = {best['mu1']:.3f}  ->  SKR = {best_skr:.3e} Hz")

plt.figure(figsize=(6,4))
plt.plot(s1.axis('mu1'), s1.metrics['skr'])
plt.plot(best['mu1'], best_skr, 'o', ms=9, label=f"optimum μ₁={best['mu1']:.2f}")
plt.xlabel('signal intensity μ₁'); plt.ylabel('secret-key rate (Hz)')
plt.title(f'Tuning μ₁ @ {LOSS_DB:.0f} dB'); plt.legend(); plt.grid(alpha=.3); plt.show()

## 3. The sensitivity surface — (μ₁, basis bias p_Z)

A 2-D sweep gives the *map* a researcher wants in their head before touching expensive hardware. The optimizer (Nelder-Mead) lands on the same peak the grid shows.

In [ ]:
s2 = sweep(run, {'mu1': list(np.round(np.linspace(0.15,0.9,25),4)),
                 'p_Z': list(np.round(np.linspace(0.5,0.97,25),4))})
opt = optimize(run, {'mu1': (0.15,0.9), 'p_Z': (0.5,0.97)}, 'skr', maximize=True, seed=1)
print('optimizer:', {k: round(v,3) for k,v in opt.best_params.items()}, '-> SKR', f'{opt.best_value:.3e} Hz')

X, Y, Z = s2.surface('mu1', 'p_Z', 'skr')
plt.figure(figsize=(6.5,4.5))
plt.contourf(X, Y, np.where(Z>0, Z, np.nan), levels=20, cmap='viridis')
plt.colorbar(label='SKR (Hz)')
plt.plot(opt.best_params['mu1'], opt.best_params['p_Z'], '*', color='red', ms=16)
plt.xlabel('signal intensity μ₁'); plt.ylabel('basis bias p_Z')
plt.title('SKR sensitivity surface'); plt.show()

## 4. A control-loop choice that reaches the security proof

Turn the AMZI phase-lock **off** and the phase wanders → the X-basis error inflates → the single-photon phase-error bound rises → the *proven* key shrinks (and dies at distance). This is the device stack feeding the security proof end-to-end.

In [ ]:
for locked in (True, False):
    _g, sched, det, _a = build_decoy_bb84(prof, length_km=40.0, locked=locked)
    sched.run(n_ticks=300, dt_slow=1e-3, pulses_per_tick=150_000, rng=np.random.default_rng(1))
    r = det.rates()
    s, kr = skr_from_rates(r, mu1=MU1, mu2=MU2, p_mu1=P1, p_Z=PZ, n_Z_block=1e9, rep_rate=REP)
    print(f"phase-lock {'ON ' if locked else 'OFF'}:  QBER_X={r['E_X1']*100:5.2f}%   proven SKR = {s:.3e} Hz")

## 5. Same kernel, a different domain — QRNG

The QRNG plugin is built from the *same* kernel primitives (graph, scheduler, Fock backend, dark counts) with **no new kernel code**. Its metric is min-entropy, and detector imperfection (efficiency mismatch) lowers the extractable randomness — the platform generalizes beyond QKD.

In [ ]:
from qsim.qrng.reference import run_qrng

deltas = np.linspace(0.0, 0.08, 9)
hmin = [run_qrng(eta_a=0.20+d, eta_b=0.20-d, seed=1)['min_entropy'] for d in deltas]
plt.figure(figsize=(6,4))
plt.plot(deltas*2, hmin, 'o-')
plt.xlabel('detector efficiency mismatch |η_A - η_B|'); plt.ylabel('min-entropy (bits/bit)')
plt.title('QRNG randomness vs detector imbalance'); plt.grid(alpha=.3); plt.ylim(0,1.02); plt.show()
print('balanced H_min =', round(hmin[0],4), 'bits/bit;  most-mismatched =', round(hmin[-1],4))

---
That is the loop: **build → run → probe → sweep/optimize → read a validated metric**, across two domains, all on a laptop. See `demos/` for the scripted versions and `scenarios/` for declarative, shareable experiment files.